# 🔍 Detecção de Anomalias em Transações Financeiras
> **Projeto DIO — Bootcamp Afya / Accenture Dados**

Neste notebook construímos um **detector de fraudes bancárias** inspirado nos sistemas anti-fraude usados por empresas como Nubank. Utilizamos três algoritmos de Machine Learning não-supervisionados:

| Algoritmo | Abordagem |
|---|---|
| **Isolation Forest** | Isola anomalias em árvores de decisão aleatórias |
| **Local Outlier Factor (LOF)** | Compara densidade local de vizinhos |
| **Z-Score** | Desvio padrão por feature |
| **Ensemble** | Votação majoritária dos 3 modelos |

## 📦 1. Imports e Configuração

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler
from scipy import stats

import warnings
warnings.filterwarnings('ignore')

# Importa módulos do projeto
from anomaly_detector import (
    gerar_transacoes, preprocessar, FEATURES,
    detectar_isolation_forest, detectar_lof, detectar_zscore,
    calcular_metricas
)

print('✅ Ambiente configurado com sucesso!')

## 📊 2. Geração do Dataset Sintético

Simulamos transações bancárias com padrões realistas:
- **Transações normais**: valores medianos, horário comercial, baixa frequência, distância curta
- **Fraudes**: valores extremos, madrugada, alta frequência, distância longa

In [ ]:
df = gerar_transacoes(n_normais=950, n_fraudes=50, seed=42)
print(f'Shape: {df.shape}')
print(f"\nDistribuição de classes:")
print(df['is_fraude_real'].value_counts().rename({0: 'Normal', 1: 'Fraude'}))
df.head(10)

In [ ]:
df.describe().round(2)

## 🔭 3. Análise Exploratória (EDA)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
fig.suptitle('Distribuição das Features por Classe', fontsize=14, weight='bold')

features = ['valor', 'hora', 'frequencia_dia', 'distancia_km']
labels   = ['Valor (R$)', 'Hora do dia', 'Frequência/dia', 'Distância (km)']

for ax, feat, lbl in zip(axes.flat, features, labels):
    for classe, cor, nome in [(0, '#2196F3', 'Normal'), (1, '#F44336', 'Fraude')]:
        dados = df.loc[df['is_fraude_real'] == classe, feat]
        ax.hist(dados, bins=30, alpha=0.6, color=cor, label=nome, edgecolor='white')
    ax.set_title(lbl, weight='bold')
    ax.legend()

plt.tight_layout()
plt.show()

## 🤖 4. Pré-processamento e Modelagem

In [ ]:
# Normalização
X, scaler = preprocessar(df)
print(f'Features normalizadas: {FEATURES}')
print(f'Shape X: {X.shape}')

In [ ]:
# Aplica os 3 modelos
df['pred_isolation_forest'] = detectar_isolation_forest(X, contamination=0.05)
df['pred_lof']              = detectar_lof(X, contamination=0.05)
df['pred_zscore']           = detectar_zscore(df)

# Ensemble: votação majoritária
import numpy as np
votos = (
    (df['pred_isolation_forest'] == -1).astype(int)
    + (df['pred_lof']            == -1).astype(int)
    + (df['pred_zscore']         == -1).astype(int)
)
df['pred_ensemble'] = np.where(votos >= 2, -1, 1)

print('Anomalias detectadas por modelo:')
for col in ['pred_isolation_forest', 'pred_lof', 'pred_zscore', 'pred_ensemble']:
    n = (df[col] == -1).sum()
    print(f'  {col:30s}: {n:3d} ({n/len(df)*100:.1f}%)')

## 📈 5. Visualização das Detecções

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)

modelos = ['pred_isolation_forest', 'pred_lof', 'pred_zscore']
nomes   = ['Isolation Forest', 'Local Outlier Factor', 'Z-Score']

for ax, col, nome in zip(axes, modelos, nomes):
    normais   = df[df[col] == 1]
    detectados = df[df[col] == -1]
    perdidos  = df[(df[col] == 1) & (df['is_fraude_real'] == 1)]

    ax.scatter(normais['valor'],    normais['distancia_km'],
               c='#2196F3', alpha=0.3, s=15, label='Normal')
    ax.scatter(detectados['valor'], detectados['distancia_km'],
               c='#FF9800', alpha=0.8, s=60, edgecolors='k', linewidths=0.4,
               label='Anomalia detectada')
    ax.scatter(perdidos['valor'],   perdidos['distancia_km'],
               c='#F44336', marker='x', s=80, lw=1.5, label='Fraude perdida')

    ax.set_title(nome, weight='bold')
    ax.set_xlabel('Valor (R$)')

axes[0].set_ylabel('Distância (km)')
axes[0].legend(fontsize=7)
fig.suptitle('Detecção: Valor × Distância', fontsize=13, weight='bold')
plt.tight_layout()
plt.show()

## 📏 6. Avaliação dos Modelos

In [ ]:
y_real = df['is_fraude_real'].values

resultados = {}
for nome, col in [
    ('Isolation Forest', 'pred_isolation_forest'),
    ('Local Outlier Factor', 'pred_lof'),
    ('Z-Score', 'pred_zscore'),
    ('Ensemble', 'pred_ensemble'),
]:
    resultados[nome] = calcular_metricas(y_real, df[col].values)

df_metricas = pd.DataFrame(resultados).T[['Precisão', 'Recall', 'F1-Score']]
print(df_metricas.to_string())

In [ ]:
ax = df_metricas.plot(kind='bar', figsize=(10, 5), edgecolor='white',
                      color=['#42A5F5', '#EF5350', '#66BB6A'])
ax.set_title('Comparação de Métricas por Modelo', weight='bold', fontsize=13)
ax.set_ylabel('Score')
ax.set_ylim(0, 1.15)
ax.set_xticklabels(ax.get_xticklabels(), rotation=15, ha='right')

for container in ax.containers:
    ax.bar_label(container, fmt='%.0%%', fontsize=8, padding=2)

plt.tight_layout()
plt.show()

## 💾 7. Exportação dos Resultados

In [ ]:
from pathlib import Path

# Cria o diretório se não existir (compatível com execução a partir de qualquer pasta)
output_path = Path('../data/transacoes_com_previsoes.csv')
output_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(output_path, index=True)

# Exibe transações identificadas como fraude pelo ensemble
fraudes_detectadas = df[df['pred_ensemble'] == -1]
print(f'Total de anomalias detectadas pelo Ensemble: {len(fraudes_detectadas)}')
fraudes_detectadas[FEATURES + ['pred_ensemble', 'is_fraude_real']].head(10)

## 🏁 Conclusão

Construímos um pipeline completo de detecção de fraudes com:

- ✅ **Geração de dados sintéticos** realistas
- ✅ **Pré-processamento** com normalização
- ✅ **3 algoritmos de ML não-supervisionado**
- ✅ **Ensemble** por votação majoritária
- ✅ **Métricas** de avaliação (Precisão, Recall, F1)
- ✅ **Visualizações** detalhadas

> O **Ensemble** geralmente apresenta o melhor equilíbrio entre Precisão e Recall, reduzindo os pontos cegos de cada algoritmo individual.